In [ ]:
# HDB Resale ETL Pipeline

## Objective

This notebook demonstrates the ETL pipeline developed for the HDB Resale Price Dataset (January 2012 – December 2016).

The ETL pipeline performs the following tasks:

1. Read and combine all raw CSV files.
2. Perform data profiling.
3. Validate data quality.
4. Remove invalid records.
5. Remove duplicated records based on the composite key.
6. Recalculate remaining lease.
7. Generate the Resale Identifier.
8. Hash the identifier using SHA-256.
9. Export the required output datasets.

**Project Structure**

```
HDB_Project/
├── data/
├── notebook/
├── src/
├── run.py
└── README.md
```

In [ ]:
## Step 1 - Import Required Libraries

The following cell imports all modules required for the ETL pipeline.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [ ]:
import pandas as pd

from src.extract import extract_raw_data
from src.profiling import profile_data
from src.cleaning import clean_text, remove_duplicates
from src.validation import validate_data
from src.transformation import calculate_remaining_lease
from src.transformation import create_identifier
from src.hashing import hash_identifier

In [ ]:
## Step 2 - Read Raw Data

The ETL pipeline automatically reads all CSV files from the `data/raw/` directory and combines them into a single master dataset.

In [ ]:
df = extract_raw_data("../data/raw")

print(df.shape)

df.head()

In [ ]:
## Step 3 - Perform Data Profiling

The profiling process summarises the dataset, including:

- Data types
- Missing values
- Unique values
- Descriptive statistics

In [ ]:
profile = profile_data(df)

display(profile)

In [ ]:
## Step 4 - Clean Dataset

This step standardises text values by:

- Removing leading/trailing spaces
- Converting text fields to uppercase

In [ ]:
df = clean_text(df)

In [ ]:
## Step 5 - Validate Dataset

Validation rules are applied for:

- Month
- Town
- Flat Type
- Flat Model
- Storey Range

Invalid records are moved into the Failed dataset.

In [ ]:
clean_df, failed_df = validate_data(df)

print("Clean:", len(clean_df))
print("Failed:", len(failed_df))

In [ ]:
## Step 6 - Remove Duplicate Records

The composite key consists of all columns except `resale_price`.

If duplicate keys exist, the record with the higher resale price is retained.

In [ ]:
clean_df, duplicate_df = remove_duplicates(clean_df)

In [ ]:
## Step 7 - Calculate Remaining Lease

Remaining lease is recalculated based on a 99-year lease using the current execution date.

In [ ]:
clean_df = calculate_remaining_lease(clean_df)

In [ ]:
## Step 8 - Generate Resale Identifier

The Resale Identifier is generated using:

- Block Number
- Average Monthly Resale Price
- Month
- Town Initial

In [ ]:
transformed_df = create_identifier(clean_df)

In [ ]:
## Step 9 - Hash Identifier

The generated identifier is hashed using the SHA-256 algorithm.

In [ ]:
hashed_df = hash_identifier(transformed_df)

In [ ]:
## Step 10 - Save Output Files

The ETL pipeline generates the following outputs:

- Cleaned Dataset
- Failed Dataset
- Transformed Dataset
- Hashed Dataset

In [ ]:
clean_df.to_csv("../data/cleaned/clean.csv", index=False)

failed_df.to_csv("../data/failed/failed.csv", index=False)

transformed_df.to_csv("../data/transformed/transformed.csv", index=False)

hashed_df.to_csv("../data/hashed/hashed.csv", index=False)

print("Output files generated successfully.")

In [ ]:
## Summary

The ETL pipeline has completed successfully.

Generated output files:

- `data/cleaned/clean.csv`
- `data/failed/failed.csv`
- `data/transformed/transformed.csv`
- `data/hashed/hashed.csv`

This notebook demonstrates the complete ETL workflow while keeping the implementation modular within the `src` package.